# Literature GAP Matrix dengan Pipeline RAG

Notebook ini membaca beberapa paper (PDF), menyimpannya ke vector store (ChromaDB) dengan `paper_id`, lalu mengisi kolom matriks **per paper** memakai retrieval + LLM (Groq). Hasilnya berupa `gap_matrix.csv` / `.xlsx` dan draf analisis gap.

**Cara pakai (urut):**
1. Upload `paper6.pdf` dst. ke Colab (ikon folder di kiri, lalu upload ke `/content`).
2. Simpan API key Groq di **Secrets** Colab (ikon kunci di kiri) dengan nama `GROQ_API_KEY`, lalu aktifkan *Notebook access*. Jangan menulis key langsung di kode.
3. Jalankan sel dari atas ke bawah. **Ujicoba dulu dengan satu paper** (atur `HANYA_PAPER` di sel konfigurasi).
4. Hasil ekstraksi LLM hanyalah **draf**: cek tiap sel ke PDF sebelum dipakai.

In [1]:
!pip install -q pymupdf chromadb sentence-transformers openai pandas openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


## 1. Konfigurasi

In [2]:
import re, time, glob, os, json
import pymupdf
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI, RateLimitError

# ---- API key (dari Colab Secrets, bukan ditulis di kode) ----
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    import getpass
    GROQ_API_KEY = getpass.getpass("Masukkan Groq API key: ")

client_groq = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

# ---- Pengaturan ----
MODEL_LLM  = "openai/gpt-oss-120b"        # cek katalog model aktif di console.groq.com
MODEL_EMB  = "paraphrase-multilingual-mpnet-base-v2"
FOLDER_PDF = "/content"                   # lokasi PDF di Colab
POLA_FILE  = "paper*.pdf"
HANYA_PAPER = None                        # contoh uji: "paper6.pdf"  (None = semua paper)
UKURAN_CHUNK, OVERLAP, K = 1000, 150, 5
JEDA_DETIK = 15                           # jeda antar-panggilan LLM (batas token per menit Groq)

## 2. Fungsi pembaca PDF, chunking, dan DOI

In [3]:
def baca_bersih(path):
    """Baca PDF, rapikan spasi, dan buang daftar pustaka agar tidak mengotori retrieval."""
    dok = pymupdf.open(path)
    teks = " ".join(h.get_text() for h in dok)
    teks = re.sub(r"\s+", " ", teks).strip()
    idx = max(teks.rfind("References"), teks.rfind("REFERENCES"))
    if idx > 0.6 * len(teks):            # hanya dipotong jika berada di bagian akhir
        teks = teks[:idx]
    return teks

def chunking_ukuran_tetap(teks, ukuran=UKURAN_CHUNK, overlap=OVERLAP):
    chunks, mulai = [], 0
    while mulai < len(teks):
        chunks.append(teks[mulai:mulai + ukuran])
        mulai += ukuran - overlap
    return chunks

def ambil_doi(path):
    """DOI dicari dengan regex hanya di halaman 1 (agar tidak tertukar dengan DOI daftar pustaka)."""
    h1 = pymupdf.open(path)[0].get_text()
    h1 = re.sub(r"\s+", " ", h1)
    m = re.search(r"10\.\d{4,9}/[^\s]+", h1)
    return m.group(0).rstrip(".,;)") if m else "tidak ditemukan (isi manual)"

def teks_halaman_awal(path, n_char=2000):
    return re.sub(r"\s+", " ", pymupdf.open(path)[0].get_text())[:n_char]

## 3. Bangun vector store (satu koleksi, setiap chunk membawa `paper_id`)

In [4]:
files = sorted(glob.glob(os.path.join(FOLDER_PDF, POLA_FILE)),
               key=lambda f: int(re.search(r"(\d+)", os.path.basename(f)).group(1)))
if HANYA_PAPER:
    files = [f for f in files if os.path.basename(f) == HANYA_PAPER]
print("Paper yang diproses:", [os.path.basename(f) for f in files])

model_embedding = SentenceTransformer(MODEL_EMB)
client_vektor = chromadb.Client()
if "paper_matrix" in [c.name for c in client_vektor.list_collections()]:
    client_vektor.delete_collection("paper_matrix")
koleksi = client_vektor.create_collection(name="paper_matrix")

ringkasan = []
for f in files:
    pid = os.path.basename(f)
    chunks = chunking_ukuran_tetap(baca_bersih(f))
    koleksi.add(documents=chunks,
                embeddings=model_embedding.encode(chunks).tolist(),
                metadatas=[{"paper_id": pid}] * len(chunks),
                ids=[f"{pid}_{i}" for i in range(len(chunks))])
    ringkasan.append({"paper": pid, "jumlah_chunk": len(chunks), "DOI": ambil_doi(f)})

pd.DataFrame(ringkasan)   # cek awal: jumlah chunk dan DOI tiap paper

Paper yang diproses: ['paper6.pdf', 'paper7.pdf', 'paper8.pdf', 'paper9.pdf', 'paper10.pdf']


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,paper,jumlah_chunk,DOI
0,paper6.pdf,55,10.1016/j.eswa.2026.132388
1,paper7.pdf,30,tidak ditemukan (isi manual)
2,paper8.pdf,97,10.1016/j.ssaho.2026.103485
3,paper9.pdf,105,10.1016/j.ssaho.2026.103135
4,paper10.pdf,114,10.1016/j.imu.2026.101810


## 4. Fungsi ekstraksi per kolom (retrieval terfilter per paper + LLM)

In [5]:
# kolom: (instruksi untuk LLM, kata kunci retrieval dalam bahasa Inggris+Indonesia)
KOLOM = {
 "PROBLEM":      ("masalah atau gap penelitian yang ingin diselesaikan penulis",
                  "problem statement, motivation, research gap, challenge addressed, introduction, masalah penelitian"),
 "METODE":       ("metode yang diusulkan: arsitektur, retriever/embedding, LLM, dan pipeline",
                  "proposed method, architecture, retriever, embedding model, language model, methodology, metode"),
 "DATASET":      ("dataset atau korpus yang dipakai beserta ukurannya",
                  "dataset, corpus, data collection, number of documents, benchmark, evaluation data, dataset yang digunakan"),
 "TEMUAN":       ("hasil/temuan utama beserta angka metriknya",
                  "results, findings, accuracy, precision, recall, F1, performance comparison, conclusion, hasil utama"),
 "KETERBATASAN": ("keterbatasan yang DINYATAKAN penulis (bukan opini kamu)",
                  "limitations, weaknesses, threats to validity, constraints, keterbatasan penelitian"),
 "FUTURE WORK":  ("arah penelitian lanjutan yang dinyatakan penulis",
                  "future work, future research directions, further research, recommendations, conclusion, penelitian lanjutan"),
}

_pakai_reasoning_low = True

def panggil_llm(prompt):
    """Panggil Groq dengan retry otomatis jika kena rate limit (429)."""
    global _pakai_reasoning_low
    for percobaan in range(5):
        try:
            kw = {"extra_body": {"reasoning_effort": "low"}} if _pakai_reasoning_low else {}
            r = client_groq.chat.completions.create(
                model=MODEL_LLM, messages=[{"role": "user", "content": prompt}], **kw)
            return (r.choices[0].message.content or "").strip()
        except RateLimitError as e:
            tunggu = re.search(r"try again in ([\d.]+)s", str(e))
            detik = float(tunggu.group(1)) + 3 if tunggu else 25
            print(f"  rate limit, menunggu {detik:.0f} detik...")
            time.sleep(detik)
        except Exception as e:
            if _pakai_reasoning_low:           # kalau parameter reasoning tidak didukung, coba tanpa itu
                _pakai_reasoning_low = False
                continue
            print("  error:", e)
            return "GAGAL - isi manual"
    return "GAGAL - isi manual"

def isi_kolom(pid, kolom, instruksi, kunci, k=K):
    q = model_embedding.encode([kunci]).tolist()
    ctx = koleksi.query(query_embeddings=q, n_results=k, where={"paper_id": pid})["documents"][0]
    prompt = (
        f"Berikut potongan teks dari SATU paper ilmiah. Tuliskan kolom {kolom}: {instruksi}.\n"
        "Aturan: maksimal 3 kalimat, bahasa Indonesia, sertakan angka bila ada. "
        "Gunakan hanya informasi pada KONTEKS. Jika tidak ada pernyataan eksplisit, tulis "
        "\"tidak disebutkan eksplisit\". Jangan menebak.\n\nKONTEKS:\n" + "\n\n".join(ctx))
    return panggil_llm(prompt)

def ambil_judul(path):
    prompt = ("Dari teks halaman pertama paper berikut, tulis HANYA judul lengkap paper "
              "(tanpa penjelasan tambahan).\n\nTEKS:\n" + teks_halaman_awal(path))
    return panggil_llm(prompt)

## 5. Jalankan pipeline (± 1,5–2 menit per paper; jalankan dulu dengan 1 paper)

In [6]:
baris = []
for f in files:
    pid = os.path.basename(f)
    nomor = re.search(r"(\d+)", pid).group(1)
    row = {"PAPER": f"PAPER {nomor}", "DOI": ambil_doi(f)}
    for kolom, (instruksi, kunci) in KOLOM.items():
        row[kolom] = isi_kolom(pid, kolom, instruksi, kunci)
        time.sleep(JEDA_DETIK)
    row["JUDUL (tambahan)"] = ambil_judul(f)
    time.sleep(JEDA_DETIK)
    baris.append(row)
    print(pid, "selesai")

df = pd.DataFrame(baris)
pd.set_option("display.max_colwidth", 200)
df

paper6.pdf selesai
paper7.pdf selesai
paper8.pdf selesai
paper9.pdf selesai
paper10.pdf selesai


,PAPER,DOI,PROBLEM,METODE,DATASET,TEMUAN,KETERBATASAN,FUTURE WORK,JUDUL (tambahan)
0,PAPER 6,10.1016/j.eswa.2026.132388,**PROBLEM:** tidak disebutkan eksplisit.,**METODE:** Sistem menggunakan arsitektur RAG yang menggabungkan modul retrieval berbasis ESBERT dengan modul generasi berbasis LLM pada antarmuka Streamlit; retriever/embedding memakai model ESBE...,**DATASET:** tidak disebutkan eksplisit.,"**Temuan:** ESBERT menghasilkan precision tertinggi di antara semua model (93.21 % pada Gale Medical Encyclopedia, 93.04 % pada MedMCQA, 92.83 % pada MedQA, dan 92.46 % pada BioASQ) serta recall y...","**KETERBATASAN:** \n1. Untuk penerapan yang dapat diandalkan, diperlukan re‑indeksasi korpus secara kontinu, desain yang memperhatikan privasi, kemampuan multibahasa, serta validasi klinis oleh p...",tidak disebutkan eksplisit.,Retrieval augmented generation (RAG) with enhanced SBERT fine-tuning vector summarization in medical domain
1,PAPER 7,tidak ditemukan (isi manual),"**PROBLEM:** \nPenelitian RAG dalam konteks pendidikan masih didominasi oleh bahasa sumber daya tinggi (terutama bahasa Inggris), sehingga terdapat kekurangan studi yang fokus pada bahasa ber‑sum...",**METODE:** \nArsitektur yang diusulkan adalah sistem Retrieval‑Augmented Generation (RAG) yang menggabungkan modul pengambilan dokumen dengan model bahasa generatif. Retriever menggunakan embedd...,DATASET: tidak disebutkan eksplisit.,**TEMUAN:** \n- Pendekatan hibrida dapat meningkatkan akurasi pipeline hingga **4 %**. \n- Koutsiaki et al. menemukan akurasi tertinggi pada chatbot RAG berbahasa Yunani ketika menggunakan konte...,"- Penulis menyatakan bahwa sistem RAG masih sulit dibuat cepat dan murah tanpa mengorbankan akurasi, rentan terhadap perubahan data, serta rawan eksploitasi keamanan. \n- Keterbatasan lain melipu...","Future work sebaiknya difokuskan pada pengembangan RAG untuk lingkungan bahasa ber‑sumber daya rendah, termasuk adaptasi multibahasa dan lintas‑domain serta evaluasi keamanan dan privasi sistem. P...",Retrieval-Augmented Generation (RAG) Learning Systems: Revolutionizing AI‑Powered Education
2,PAPER 8,10.1016/j.ssaho.2026.103485,"**PROBLEM:** Penelitian ini tidak melakukan penilaian empiris terhadap hasil pedagogis seperti pemahaman belajar, perubahan perilaku, atau kegunaan instruksional, sehingga masih belum diketahui ef...","**METODE:** Sistem menggunakan arsitektur RAG yang menggabungkan lima kriteria (mis. status indeks domain, verifikasi allowlist, kesamaan semantik) untuk menghasilkan skor risiko dan penjelasan be...",**DATASET:** tidak disebutkan eksplisit.,"- Claude Sonnet 3.7 menunjukkan konsistensi tertinggi dengan koefisien variasi terendah (CV ≈ 0.0975) dan ICC = 0.980 (95 % CI [0.964, 0.991]), lebih baik dibandingkan GPT o1 (ICC = 0.920) dan Dee...","Keterbatasan yang dinyatakan penulis meliputi: (1) penjelasan yang dihasilkan LLM bersifat probabilistik dan belum dievaluasi secara sistematis, sehingga berisiko mengandung halusinasi atau infere...","Future research harus melakukan evaluasi empiris terhadap efektivitas pendidikan sistem yang diusulkan, termasuk pengukuran pemahaman pembelajar, perubahan perilaku, dan kegunaan instruksional. Pe...",Smishing risk assessment using RAG and its potential for pedagogical recontextualization for cybersecurity education
3,PAPER 9,10.1016/j.ssaho.2026.103135,**PROBLEM:** \nPenelitian belum memecahkan masalah kualitas dan struktur basis pengetahuan pada sistem Retrieval‑Augmented Generation (RAG) yang dapat menyebabkan respons chatbot tidak akurat ata...,"**METODE:** \nArsitektur RAG chatbot menghubungkan tiga lapisan — input siswa di LMS, retrieval dari basis pengetahuan yang selaras kurikulum, dan output umpan balik untuk revisi tulisan. Retriev...",**DATASET:** tidak disebutkan eksplisit.,"**TEMUAN:** \n1. Pada pre‑test tidak ada perbedaan signifikan antara kelompok kontrol (M = 7.94) dan eksperimental (M = 8.21); t(68) = −1.08, p = 0.286, 95 % CI [−0.78, 0.23]. \n2. Pada post‑tes...",- Pen

## 6. Simpan hasil (impor ke Google Sheets *TUGAS RAG KELOMPOK*)

In [7]:
df.to_csv("gap_matrix.csv", index=False, encoding="utf-8-sig")
df.to_excel("gap_matrix.xlsx", index=False)
try:
    from google.colab import files as colab_files
    colab_files.download("gap_matrix.xlsx")
except Exception:
    print("Tersimpan: gap_matrix.csv dan gap_matrix.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. (Opsional) Draf analisis gap dalam bentuk paragraf

Jalankan **setelah** kalian memeriksa dan memperbaiki isi matriks. Hasilnya hanya draf: edit dan sesuaikan dengan topik usulan paper kelompok.

In [8]:
TOPIK_USULAN = "RAG untuk asisten pembelajaran/perkuliahan mahasiswa"   # sesuaikan

tabel_teks = df.drop(columns=["JUDUL (tambahan)"], errors="ignore").to_csv(index=False)
prompt_gap = f"""Kamu membantu mahasiswa S2 menulis analisis literature gap. Berikut matriks literatur
(CSV) tentang RAG. Tulis 3 paragraf bahasa Indonesia akademik untuk usulan paper tentang: {TOPIK_USULAN}.
Paragraf 1: pola umum dan kekuatan penelitian yang ada.
Paragraf 2: gap yang terlihat dari kolom KETERBATASAN dan FUTURE WORK serta kolom yang seragam atau kosong.
Paragraf 3: usulan kontribusi paper kelompok yang menjawab gap tersebut.
Gunakan HANYA informasi pada matriks, sebut paper dengan namanya (mis. PAPER 6), dan jangan menambah fakta baru.

MATRIKS:
{tabel_teks}"""

time.sleep(20)
analisis = panggil_llm(prompt_gap)
print(analisis)
open("analisis_gap.txt", "w", encoding="utf-8").write(analisis)

**Paragraf 1 – Pola umum dan kekuatan penelitian yang ada**  
Sebagian besar karya yang tercatat dalam matriks meneliti arsitektur Retrieval‑Augmented Generation (RAG) dengan fokus pada optimalisasi komponen retrieval (mis. ESBERT pada *Paper 6*, dense/BM25 pada *Paper 10*) serta evaluasi kinerja LLM (GPT‑o1, DeepSeek V3, Claude Sonnet 3.7 pada *Paper 8*). Hasil empiris menunjukkan peningkatan metrik teknis yang signifikan, seperti precision > 93 % dan F1‑score ≈ 94 % pada *Paper 6*, serta akurasi tambahan sebesar +0.034 pada konfigurasi CAF‑RAG di *Paper 10*. Beberapa studi juga melaporkan manfaat pedagogis awal, misalnya peningkatan skor post‑test pada chatbot RAG untuk umpan‑balik tulisan (*Paper 9*) dan peningkatan akurasi pipeline hingga 4 % pada skenario pendidikan multibahasa (*Paper 7*). Secara keseluruhan, literatur yang ada menegaskan bahwa kombinasi retriever yang terlatih secara khusus dengan LLM dapat memperbaiki kualitas respons dan memberikan dasar yang kuat untuk pengem

3655